# Component RM with native PSRSALSA rmsynth
Python identifies pulse windows from Stokes I. For each component it passes the original archive and inclusive bin ranges to the OzSTAR executable. All RM spectra, instrumental response fits and uncertainty estimates come from that executable. The native RM map and spectrum are saved as PostScript plots. Inspect the gate plot before interpreting the results.

In [ ]:
from pathlib import Path
import hashlib
import json
import shlex
import subprocess
import tempfile
import numpy as np
import matplotlib.pyplot as plt

CHECKOUT = Path("/fred/oz002/skavya/Working_Dir/scripts/psrsalsa")
BINARY = CHECKOUT / "bin/rmsynth"
REVISION = "2efbaf4592e08fdb76002bbc9dc25e8e4441813b"
INPUT = Path("/fred/oz341/B1259_kavya/parkes/high_res/test_archives_rfi/sub/ext")
EXTENSION = ".center"
OUTPUT = Path("native_psrsalsa_rm")
RM_LOW, RM_HIGH, RM_STEPS = -10000., 10000., 20000
FTOL = 0.001
BOOTSTRAP = 0  # Native default. Set to e.g. 500 for Gaussian-bootstrap errors.
LONGITUDE_FITS = False  # Also run -long for each component
FIX_SEED = True  # -fixseed gives repeatable native draws
PEAK_SIGMA = 3.0
SMOOTH_PHASE = 0.004

## Identify the components
Use a dedispersed, time-integrated intensity profile at the archive’s native phase resolution. The smoothed profile is searched cyclically for up to two significant peaks. A sufficiently deep valley separates neighbouring peaks. A single peak is labelled single. Two are labelled leading and trailing by increasing archive phase. Separate inclusive ranges represent a gate crossing bin zero. The union of all detected components supplies the signal window for the native noise calculation.

In [ ]:
def detect(profile):
    n = len(profile)
    width = max(0.7, n * SMOOTH_PHASE)
    radius = max(1, round(4*width))
    kernel = np.exp(-0.5*(np.arange(-radius, radius+1)/width)**2)
    smooth = np.convolve(np.pad(profile, radius, mode="wrap"), kernel/kernel.sum(), mode="valid")
    origin = int(smooth.argmin())
    y = np.roll(smooth, -origin)
    baseline = np.median(y)
    noise = max(1.4826 * np.median(abs(y - baseline)), np.finfo(float).tiny)
    candidates = np.flatnonzero((y > np.roll(y, 1)) & (y >= np.roll(y, -1))
                                & (y > baseline + PEAK_SIGMA*noise))
    selected = []
    for peak in candidates[np.argsort(y[candidates])[::-1]]:
        if not selected:
            selected.append(peak)
            continue
        a, b = sorted([selected[0], peak])
        # A separate component needs a resolved valley on both circular paths.
        saddle = max(y[a:b+1].min(), min(y[:a+1].min(), y[b:].min()))
        if min(b-a, n-b+a) >= max(2, round(3*width)) and y[peak]-saddle >= PEAK_SIGMA*noise:
            selected.append(peak)
            break
    chosen = sorted(selected)
    gates = []
    for p in chosen:
        left, right = p, p
        while left > 0 and y[left-1] > baseline + 2*noise:
            left -= 1
        while right < n-1 and y[right+1] > baseline + 2*noise:
            right += 1
        if len(chosen) == 2:
            a, b = chosen
            valley = a + np.argmin(y[a:b+1])
            if p == a:
                right = min(right, valley)
            else:
                left = max(left, valley+1)
        gates.append(np.roll((np.arange(n) >= left) & (np.arange(n) <= right), origin))
    gates.sort(key=lambda gate: np.argmax(np.where(gate, smooth, -np.inf)))
    return gates

In [ ]:
def intensity(path):
    import psrchive
    archive = psrchive.Archive_load(str(path))
    if archive.get_npol() != 4 or archive.get_nsubint() != 1:
        raise ValueError("Prepare a single-subintegration, full-Stokes archive for rmsynth")
    if archive.get_faraday_corrected():
        raise ValueError("Use an archive before Faraday correction")
    archive.convert_state("Stokes")
    archive.dedisperse()
    data = archive.get_data()[0, 0]
    weights = archive.get_weights()[0]
    good = (weights > 0) & np.isfinite(weights) & np.isfinite(data).all(axis=1)
    if not good.any():
        raise ValueError("No usable intensity channels")
    return np.average(data[good], axis=0, weights=weights[good])

def intervals(mask):
    edges = np.diff(np.r_[False, mask, False].astype(int))
    return list(zip(np.flatnonzero(edges == 1).tolist(),
                    (np.flatnonzero(edges == -1) - 1).tolist()))

def command(gate, signal, longitude, filename):
    args = [str(BINARY.resolve()), "-rm", f"{RM_LOW} {RM_HIGH} {RM_STEPS}",
            "-ftol", str(FTOL), "-ascii", "-v", "-nocounters", "-stokes",
            "-device", "spectrum.ps/CPS", "-device2", "map.ps/CPS", "-device3", "/null"]
    for option, mask in [("-onpulse", gate), ("-onpulse2", signal)]:
        for left, right in intervals(mask):
            args.extend([option, f"{left} {right}"])
    if BOOTSTRAP:
        args.extend(["-bootstrap", str(BOOTSTRAP)])
    if FIX_SEED:
        args.append("-fixseed")
    if longitude:
        args.append("-long")
    return args + [filename]

## Run rmsynth
Each invocation gets a fresh directory and a link to the original archive. PSRSALSA writes its RMtable beside that link. The notebook checks its header, row count and (for longitude fits) the bin numbers. The native log contains warnings and diagnostics. No peak refinement or error calculation is performed in Python.

In [ ]:
def native_fit(path, folder, gate, signal, longitude):
    folder.mkdir()
    link = folder / "input.ar"
    link.symlink_to(path.resolve())
    args = command(gate, signal, longitude, link.name)
    (folder / "command.txt").write_text(shlex.join(args) + "\n")
    log = folder / "rmsynth.log"
    with log.open("w") as out:
        result = subprocess.run(args, cwd=folder, stdout=out, stderr=subprocess.STDOUT,
                                stdin=subprocess.DEVNULL)
    table = folder / "input.RMtable"
    if result.returncode or not table.is_file():
        raise RuntimeError(f"No complete native result. See {log}")
    expected = (["binnr", "Profile"] if longitude else []) + ["RM", "RMerror(analytic)"]
    if BOOTSTRAP:
        expected.append("RMerror(bootstrap)")
    with table.open() as stream:
        header = stream.readline().lstrip("#").split()
    values = np.loadtxt(table, ndmin=2)
    if header != expected or values.shape != (int(gate.sum()) if longitude else 1, len(expected)):
        raise ValueError(f"Unexpected native table format: {table}")
    if longitude and not np.array_equal(values[:, 0], np.flatnonzero(gate)):
        raise ValueError(f"Native longitude bins differ from supplied gate: {table}")
    print(folder.name, table, header, values[0] if not longitude else f"{len(values)} bins")

def analyse(path, root):
    folder = root / path.name
    folder.mkdir()
    profile = intensity(path)
    gates = detect(profile)
    labels = ["single"] if len(gates) == 1 else ["leading", "trailing"]
    regions = {name: intervals(gate) for name, gate in zip(labels, gates)}
    (folder / "gates.json").write_text(json.dumps(regions, indent=2))
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.plot(profile, color="black", lw=1)
    for name, ranges in regions.items():
        for i, (a, b) in enumerate(ranges):
            ax.axvspan(a-.5, b+.5, alpha=.25, color="C0" if name != "trailing" else "C1",
                       label=name if i == 0 else None)
    if gates:
        ax.legend()
    ax.set(xlabel="Archive phase bin", ylabel="Stokes I", title=path.name)
    fig.tight_layout()
    fig.savefig(folder / "gates.png")
    plt.show()
    plt.close(fig)
    if not gates:
        print(path.name, "No significant component; see gates.png")
        return
    signal = np.logical_or.reduce(gates)
    if signal.all():
        raise ValueError("No off-pulse bins remain for native uncertainty estimates")
    for name, gate in zip(labels, gates):
        native_fit(path, folder / name, gate, signal, False)
        if LONGITUDE_FITS:
            native_fit(path, folder / (name + "_long"), gate, signal, True)

In [ ]:
if not RM_LOW < RM_HIGH or RM_STEPS < 2 or BOOTSTRAP < 0 or FTOL <= 0:
    raise ValueError("Check the native RM grid, bootstrap count and fit tolerance")
revision = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
if revision != REVISION:
    raise ValueError(f"Expected PSRSALSA {REVISION}, found {revision}")
digest = hashlib.sha256(BINARY.read_bytes()).hexdigest()
files = [INPUT] if INPUT.is_file() else sorted(p for p in INPUT.glob("*" + EXTENSION) if p.is_file())
if not files:
    raise FileNotFoundError(f"No matching archives at {INPUT}")
OUTPUT.mkdir(parents=True, exist_ok=True)
root = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT)).resolve()
(root / "binary.json").write_text(json.dumps(dict(checkout=revision, binary=str(BINARY.resolve()),
                                                 sha256=digest), indent=2))
print("Run directory:", root)
errors = {}
for path in files:
    try:
        analyse(path, root)
    except (OSError, RuntimeError, ValueError) as exc:
        errors[str(path)] = str(exc)
        print(path.name, exc)
if errors:
    (root / "errors.json").write_text(json.dumps(errors, indent=2))
    raise RuntimeError(f"{len(errors)} archive(s) failed. See {root / 'errors.json'}")